# Automotive Technical Design Generator

This notebook demonstrates an agentic system which generates comprehensive technical design documents from validated business requirements document (BRD), software requirements document (SRD) and other supplementary documents. The agent is implemented using Strands agents deployed on Amazon Bedrock AgentCore Runtime with inbound JWT authentication and integrates with AgentCore Gateway to access design guidelines from S3.

## Prerequisites

To execute this tutorial you will need:
* Python 3.13+ (Recommended: virtual environment at workspace level created and top level requirements installed)
* AWS credentials configured for a US based region

In [ ]:
# Optionally set your AWS_PROFILE and AWS_REGION (region has to be US based)
import os
# os.environ['AWS_PROFILE'] = 'default'
# os.environ['AWS_REGION'] = 'us-west-2'

In [ ]:
# Optionally check that required packages are installed
!pip install --force-reinstall -r ../design-agent/requirements.txt --quiet

## Deploy Design Guidelines Infrastructure

The design guidelines infrastructure (S3 bucket, Lambda, IAM roles) is deployed via the CDK stack. Ensure the MA3T CDK stack has been deployed before proceeding.

In [ ]:
# Retrieve CDK stack outputs for the design guidelines infrastructure.
# The AutomotiveVCycleStack is deployed as a nested stack within the MA3T CDK deployment.
# We search CloudFormation stacks for the expected output keys.

import boto3
from boto3.session import Session

boto_session = Session()
_cfn_region = boto_session.region_name
cfn_client = boto3.client('cloudformation', region_name=_cfn_region)

stack_outputs = {}
paginator = cfn_client.get_paginator('describe_stacks')
for page in paginator.paginate():
    for stack in page['Stacks']:
        outputs = {o['OutputKey']: o['OutputValue'] for o in stack.get('Outputs', [])}
        if 'GuidelinesLambdaArn' in outputs and 'GatewayRoleArn' in outputs:
            # Map CDK output keys to the names used by this notebook
            stack_outputs = {
                'S3BucketName': outputs.get('GuidelinesBucketName'),
                'LambdaFunctionArn': outputs.get('GuidelinesLambdaArn'),
                'GatewayRoleArn': outputs.get('GatewayRoleArn'),
            }
            print(f"Found CDK stack: {stack['StackName']}")
            break
    if stack_outputs:
        break

if not stack_outputs:
    raise RuntimeError(
        "Could not find deployed CDK stack outputs. "
        "Ensure the MA3T CDK stack has been deployed (cd ma3t && ./deploy_cdk.sh)."
    )

print(f"S3 Bucket: {stack_outputs.get('S3BucketName')}")
print(f"Lambda Function ARN: {stack_outputs.get('LambdaFunctionArn')}")
print(f"Gateway Role ARN: {stack_outputs.get('GatewayRoleArn')}")

## Deploy the AgentCore Gateway

Now let's create the AgentCore Gateway that will provide access to design guidelines stored in S3 through Lambda functions.

In [ ]:
import importlib
import utils

# Reload the module if you changed the contents of utils since Kernel start.
importlib.reload(utils)
importlib.reload(boto3)

In [ ]:
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError
from boto3.session import Session

boto_session = Session()
REGION = boto_session.region_name

USER_POOL_NAME = "AgentCoreGatewayPool"
RESOURCE_SERVER_ID = "AgentCoreGatewayResId"
RESOURCE_SERVER_NAME = "AgentCoreGatewaResName"
CLIENT_NAME = "AgentCoreGatewayClientName"

SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "Read access"},
    {"ScopeName": "gateway:write", "ScopeDescription": "Write access"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Creating or retrieving Cognito resources...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"User Pool ID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("Resource server ensured.")

gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"Client ID: {gw_client_id}")

cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

In [ ]:
# CreateGateway with Cognito authorizer
# NOTE: stack_outputs must contain GatewayRoleArn from the CDK deployment
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [gw_client_id],
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(
    name='GuidelinesGateway',
    roleArn=stack_outputs.get('GatewayRoleArn'),
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='Lambda based GuidelinesGateway'
)
print(create_response)
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

In [ ]:
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": stack_outputs.get('LambdaFunctionArn'),
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_technical_guideline",
                        "description": "Retrieve technical design guideline",
                        "inputSchema": {
                            "type": "object",
                            "properties": {},
                            "required": []
                        }
                    },
                    {
                        "name": "list_available_guidelines",
                        "description": "List all available guideline files",
                        "inputSchema": {
                            "type": "object",
                            "properties": {},
                            "required": []
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]

targetname='GuidelineLambdaTarget'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='retrieves guidelines from S3',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

In [ ]:
token_response = utils.get_token(user_pool_id, gw_client_id, gw_client_secret, scopeString, REGION)
token = token_response["access_token"]
print("Token response:", token)

## Testing the Gateway with a simple Strands Agent

Let's test the gateway with sample requirements documents to generate a comprehensive technical design.

In [ ]:
# Load sample requirements documents for testing
import os

# Sample SRD document
sample_srd = """
# Software Requirements Document (SRD) - Weather Application

## Functional Requirements
- FR-001: The system shall display current weather conditions
- FR-002: The system shall provide 5-day weather forecast
- FR-003: The system shall support location-based weather data
- FR-004: The system shall cache weather data for offline access

## Non-Functional Requirements
- NFR-001: Response time shall be < 2 seconds
- NFR-002: System shall be available 99.9% of the time
- NFR-003: System shall support 1000 concurrent users
- NFR-004: Data shall be encrypted in transit and at rest

## Safety Requirements
- SR-001: Weather alerts shall not distract driver (ASIL-B)
- SR-002: System shall gracefully degrade if weather service unavailable
"""

# Sample BRD document
sample_brd = """
# Business Requirements Document (BRD) - Weather Application

## Business Objectives
- Provide real-time weather information to vehicle occupants
- Enhance driving safety through weather-aware navigation
- Improve user experience with personalized weather insights

## Target Users
- Primary: Vehicle drivers and passengers
- Secondary: Fleet managers

## Success Metrics
- User engagement: 80% daily active users
- Performance: < 2 second load time
- Reliability: 99.9% uptime

## Constraints
- Must integrate with existing vehicle infotainment system
- Comply with automotive safety standards (ISO 26262)
- Support multiple languages and regions
"""

print("Sample requirements documents loaded.")

In [ ]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL, headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

yourmodel = BedrockModel(
    model_id="us.amazon.nova-2-lite-v1:0",
    max_tokens=40000
)

In [ ]:
from strands import Agent
import logging


# Configure the root strands logger. Change it to DEBUG if you are debugging the issue.
logging.getLogger("strands").setLevel(logging.INFO)

# Add a handler to see the logs
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

srs_documents=sample_srd
brd_documents=sample_brd
other_documents=""

# Create comprehensive prompt with structured requirements
prompt = f"""Generate a comprehensive technical design document from the following validated requirements:

## Software Requirements Specification (SRS):
{srs_documents}

## Business Requirements Document (BRD):
{brd_documents}"""

if other_documents.strip():
    prompt += f"""

## Additional Supporting Documents:
{other_documents}"""

prompt += """

IMPORTANT: Before generating the design, retrieve and incorporate relevant design guidelines using the available tools. This will ensure your design follows established automotive standards and best practices.

Generate a complete technical design document that covers all required sections. NEVER include code in the technical design."""

system_prompt="""You are an automotive software architect specializing in vehicle infotainment and systems implemented in Kotlin programming language.

Generate comprehensive technical design documents from validated business requirements document (BRD), software requirements document (SRD) and other supplementary documents if they exist.

In your technical design document ALWAYS include:

- **Architecture Overview**: High-level system architecture with component diagram
- **Component Specifications**: Detailed specifications for each component
- **Interface Definitions**: APIs, protocols, and data contracts
- **Data Models**: Entity relationships, schemas, and constraints
- **Safety Considerations**: ISO 26262 compliance and risk mitigation
- **Implementation Guidance**: Step-by-step development approach
- **Error Handling**: Error scenarios and recovery strategies
- **Testing Strategy**: Test approach and coverage plan

Focus on modularity, and testability."""

with client:
    # Call the listTools 
    tools = client.list_tools_sync()
    # Create an Agent with the model and tools
    agent = Agent(model=yourmodel, tools=tools, system_prompt=system_prompt)
    print(f"Tools loaded in the agent are {agent.tool_names}")
    agent(prompt)

## Deploying the Agent to AgentCore Runtime

Now we'll configure and deploy our automotive technical design generator to AgentCore Runtime with inbound authentication and gateway integration.

### Setting up Amazon Cognito for Runtime InBound Authentication

Let's provision a Cognito User Pool with an App client and one test user for automotive applications.

In [ ]:
import sys
import os
import time

from utils import setup_cognito_user_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool for automotive applications...")
cognito_config = setup_cognito_user_pool()
print("Cognito setup completed")

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

discovery_url = cognito_config.get("discovery_url")
client_id = cognito_config.get("client_id")

print(f"Discovery URL: {discovery_url}")
print(f"Client ID: {client_id}")

### Configure AgentCore Runtime Deployment

In [ ]:
agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="../design-agent/mcp_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="../design-agent/requirements.txt",
    region=region,
    agent_name="design_generator_nova_2_lite",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedClients": [client_id]
        }
    }
)
print("Configuration response:", response)

### Launch Agent to AgentCore Runtime

In [ ]:
print("Launching automotive technical design generator to AgentCore Runtime...")
launch_result = agentcore_runtime.launch(
    env_vars={
        "GATEWAY_URL": gatewayURL,
        "GATEWAY_CLIENT_ID": gw_client_id,
        "GATEWAY_CLIENT_SECRET": gw_client_secret,
        "GATEWAY_USER_POOL_ID": user_pool_id,
        "RESOURCE_SERVER_ID": RESOURCE_SERVER_ID
    })
print("Launch result:", launch_result)

### Check AgentCore Runtime Status

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

print(f"Initial status: {status}")
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Status: {status}")

print(f"Final status: {status}")

## Testing the Automotive Technical Design Generator

Let's test our agentic system with sample requirements documents to generate a comprehensive technical design.

In [ ]:
test_payload = {
    "srs_documents": sample_srd,
    "brd_documents": sample_brd,
    "other_documents": "Additional context: This is a proof-of-concept weather application for automotive infotainment systems."
}

In [ ]:
# Test the deployed agent

bearer_token = reauthenticate_user(cognito_config.get("client_id"))

# Test with business requirements documents
print("\nTesting requirements analysis with weather application business documents...")

invoke_response = agentcore_runtime.invoke(
    test_payload,
    bearer_token=bearer_token,
)
print("Response:", invoke_response)

In [ ]:
import json
from IPython.display import display, Markdown

result = json.loads(invoke_response['response']) 
print("\n=== TECHNICAL DESIGN GENERATION RESULTS ===")
print(f"Execution time: {result.get('execution_time_ms', 'N/A')}ms")
print("\n=== GENERATED DESIGN ===")
display(Markdown(result.get('step1_design_generation', {}).get('design', 'No design generated')))
print("\n=== VALIDATION RESULTS ===")
validation = result.get('step2_design_validation', {})
print(f"Validated: {validation.get('validated', False)}")
print(f"Reason: {validation.get('reason', 'N/A')}")

## Cleanup (Optional)

Clean up the AgentCore Runtime and ECR repository when done testing.

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import boto3

# agentcore_control_client = boto3.client(
#     'bedrock-agentcore-control',
#     region_name=REGION
# )
# ecr_client = boto3.client(
#     'ecr',
#     region_name=REGION
# )
# runtime_delete_response = agentcore_control_client.delete_agent_runtime(
#     agentRuntimeId=launch_result.agent_id,
# )

# response = ecr_client.delete_repository(
#     repositoryName=launch_result.ecr_uri.split('/')[1].split(':')[0],
#     force=True
# )

# print("AgentCore Runtime is deleted.")

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# import os

# if os.path.exists('.bedrock_agentcore.yaml'):
#     os.remove('.bedrock_agentcore.yaml')
#     print("Deleted .bedrock_agentcore.yaml file")

# print("Cleanup of local files done.")

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# # Delete the gateway and all its associated targets
# # The utils.delete_gateway function handles both target deletion and gateway deletion
# # Replace the gateway name if you adjusted it during the deployment.

# gateways = agentcore_control_client.list_gateways()

# # Iterate through all pages of gateways (handles pagination)
# gateway_name="GuidelinesGateway"
# while True:
#     for gateway in gateways["items"]:
#         if gateway["name"] == gateway_name:
#             utils.delete_gateway(agentcore_control_client, gateway["gatewayId"])

#     if "nextToken" not in gateways:
#         break
#     else:
#         gateways = agentcore_control_client.list_gateways(nextToken=gateways["nextToken"])

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# # Empty the S3 bucket before deleting the CloudFormation stack
# s3_bucket_name = stack_outputs.get('S3BucketName')

# if s3_bucket_name:
#     s3_client = boto3.client('s3')
    
#     # List and delete all objects in the bucket
#     paginator = s3_client.get_paginator('list_objects_v2')
#     pages = paginator.paginate(Bucket=s3_bucket_name)
    
#     for page in pages:
#         if 'Contents' in page:
#             objects = [{'Key': obj['Key']} for obj in page['Contents']]
#             s3_client.delete_objects(Bucket=s3_bucket_name, Delete={'Objects': objects})
    
#     # List and delete all object versions (for versioned buckets)
#     paginator = s3_client.get_paginator('list_object_versions')
#     pages = paginator.paginate(Bucket=s3_bucket_name)
    
#     for page in pages:
#         objects = []
#         if 'Versions' in page:
#             objects.extend([{'Key': obj['Key'], 'VersionId': obj['VersionId']} for obj in page['Versions']])
#         if 'DeleteMarkers' in page:
#             objects.extend([{'Key': obj['Key'], 'VersionId': obj['VersionId']} for obj in page['DeleteMarkers']])
        
#         if objects:
#             s3_client.delete_objects(Bucket=s3_bucket_name, Delete={'Objects': objects})

In [ ]:
print('Cleanup is commented out. Uncomment and run to cleanup created resources')

# # Delete the CloudFormation stack
# # If you deployed via CDK, use 'cdk destroy' instead
# # from deploy_guidelines_stack import delete_design_guidelines_stack
# # success = delete_design_guidelines_stack()
# # if success:
# #     print("Stack deleted successfully")